In [1]:
# ============================================================
# exp_20260427_058_mikhail_cat2_8_repro
# Purpose:
#   Mikhail shared CAT2_8 を再学習して OOF/pred npy 化
#   S3+057+Mikhail LR blend 用素材を作る
# ============================================================

In [2]:
# 必要なら有効化
!pip install -q scikit-learn==1.7.2 catboost==1.2.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 16.3 MB/s eta 0:00:00


In [3]:
import os
import json
import gc
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from pandas.api.types import CategoricalDtype
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import differential_evolution

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260427_058_mikhail_cat2_8_repro"
    EXP_NAME = "mikhail_cat2_8_repro"

    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"

    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_SPLITS = 10
    EARLY_STOP = 100

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    CAT_PARAMS = {
        "verbose": 500,
        "loss_function": "MultiClass",
        "random_state": SEED,
        "early_stopping_rounds": EARLY_STOP,
        "eval_metric": "MultiClass",
        "n_estimators": 20000,
        "learning_rate": 0.05,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
        "depth": 2,
        "min_data_in_leaf": 32,
        "l2_leaf_reg": 0.11885685483992375,
        "bagging_temperature": 0.5220357703483831,
        "random_strength": 0.33867398942718874,
    }

    REF_NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"
    REF_OOF = {
        "018": REF_NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy",
        "046b": REF_NPY_PATH + "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy",
        "025": REF_NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy",
        "056": REF_NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy",
        "057": REF_NPY_PATH + "oof_xgb046b_high_interaction_min_biased.npy",
    }

In [5]:
# ============================================================
# Utils
# ============================================================

def score_ba(y_true, proba):
    return balanced_accuracy_score(y_true, np.argmax(proba, axis=1))


def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    s = x.sum(axis=1, keepdims=True)
    return (x / np.clip(s, eps, None)).astype(np.float32)


def mean_raw_corr(a, b):
    return float(np.mean([np.corrcoef(a[:, c], b[:, c])[0, 1] for c in range(3)]))


def mean_rank_corr(a, b):
    vals = []
    for c in range(3):
        ra = pd.Series(a[:, c]).rank(method="average").to_numpy()
        rb = pd.Series(b[:, c]).rank(method="average").to_numpy()
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))


In [6]:
# ============================================================
# Load
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH, index_col=CFG.ID_COL)
test = pd.read_csv(CFG.TEST_PATH, index_col=CFG.ID_COL)
sample = pd.read_csv(CFG.SAMPLE_PATH)

orig = pd.read_csv(CFG.ORIG_PATH)
orig = orig[train.columns]

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)
orig[CFG.TARGET] = orig[CFG.TARGET].map(CFG.TARGET_MAP)

y = train[CFG.TARGET].astype(int).reset_index(drop=True)
X_base = train.drop(columns=[CFG.TARGET]).reset_index(drop=True)
test_base = test.reset_index(drop=True)
orig = orig.reset_index(drop=True)

print(train.shape, test.shape, orig.shape)

(630000, 20) (270000, 19) (10000, 20)


In [7]:
# ============================================================
# Preprocessing
# ============================================================

class Preprocessing:
    def __init__(self, X, y, test, orig):
        self.X = X.copy()
        self.y = y.copy()
        self.test = test.copy()
        self.orig = orig.copy()

    def fit_transform(self):
        self.num_features_ = self.X.select_dtypes(exclude=["object", "bool", "category"]).columns.tolist()
        self.cat_features_ = self.X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

        self._fit_orig_target_stats(self.cat_features_ + self.num_features_)
        self._fit_feature_engineering(self.X)

        X = self._apply_orig_target_stats(self.X)
        T = self._apply_orig_target_stats(self.test)

        X = self._apply_feature_engineering(X)
        T = self._apply_feature_engineering(T)

        self._fit_frequency_encoding(X)
        X = self._apply_frequency_encoding(X)
        T = self._apply_frequency_encoding(T)

        self.num_features = X.select_dtypes(exclude=["object", "bool", "category"]).columns.tolist()
        self.cat_features = X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

        self._cat_types = {
            col: CategoricalDtype(categories=X[col].astype("string").fillna("__NA__").unique())
            for col in self.cat_features
        }

        X = self._encode_cat_codes(X)
        T = self._encode_cat_codes(T)

        self.num_features = X.select_dtypes(exclude=["category"]).columns.tolist()
        self.cat_features = X.select_dtypes(include=["category"]).columns.tolist()

        return X, self.y, T, self.cat_features, self.num_features

    def _fit_orig_target_stats(self, cols):
        self._orig_global_mean = float(self.orig[CFG.TARGET].mean())
        self._orig_stats = {}
        for c in cols:
            self._orig_stats[c] = self.orig.groupby(c, observed=False)[CFG.TARGET].mean()

    def _apply_orig_target_stats(self, df):
        df = df.copy()
        for c, ser in self._orig_stats.items():
            df[f"{c}_org_mean"] = df[c].map(ser).astype(float).fillna(self._orig_global_mean)
        return df

    def _fit_feature_engineering(self, X_train):
        X_train = X_train.copy()

        self._highcard_num = [
            c for c in self.num_features_
            if X_train[c].nunique(dropna=False) > 20
        ]

        self._bin_edges = {}
        for c in self._highcard_num:
            try:
                _, bins = pd.qcut(X_train[c], q=10, retbins=True, duplicates="drop")
                bins = np.unique(bins)
                if len(bins) >= 3:
                    self._bin_edges[c] = bins
            except Exception:
                pass

        M = X_train[self.num_features_].max(numeric_only=True)
        self._round_decimals = {}
        for c in self.num_features_:
            m = float(M.get(c, np.nan))
            if not np.isfinite(m):
                self._round_decimals[c] = None
            elif m < 10:
                self._round_decimals[c] = 3
            elif m < 100:
                self._round_decimals[c] = 2
            else:
                self._round_decimals[c] = 1

        self._comb_pairs = []
        df_str = X_train[self.cat_features_].astype(str)
        card = df_str.nunique(dropna=False)
        valid_cols = [c for c in df_str.columns if card[c] <= len(df_str) // 2]

        for c1, c2 in combinations(valid_cols, 2):
            comb = df_str[c1] + "_" + df_str[c2]
            if comb.nunique(dropna=False) <= len(df_str) // 2:
                self._comb_pairs.append((c1, c2))

        _, dummy_cols = self._compute_logits(X_train, return_dummy_cols=True)
        self._logit_dummy_cols = dummy_cols

    def _apply_feature_engineering(self, df):
        df = df.copy()

        for c, bins in self._bin_edges.items():
            out = pd.cut(df[c], bins=bins, labels=False, include_lowest=True)
            df[f"{c}_bin"] = out.astype("Int16").astype("category")

        for c in self.num_features_:
            df[f"{c}_cat"] = df[c].astype("category")

        df_str = df[self.cat_features_].astype(str)
        for c1, c2 in self._comb_pairs:
            df[f"{c1}_{c2}_comb"] = (df_str[c1] + "_" + df_str[c2]).astype("category")

        logits = self._compute_logits(df, return_dummy_cols=False)
        df = pd.concat([df, logits], axis=1)

        for c in self.num_features_:
            for k in range(-4, 4):
                df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8").astype("category")

            d = self._round_decimals.get(c)
            if d is not None:
                df[c] = df[c].round(d)

        drop = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
        df.drop(drop, axis=1, inplace=True)

        return df

    def _compute_logits(self, df, return_dummy_cols=False):
        tmp = df.copy()
        tmp["soil_lt_25"] = (tmp["Soil_Moisture"] < 25).astype(int)
        tmp["temp_gt_30"] = (tmp["Temperature_C"] > 30).astype(int)
        tmp["rain_lt_300"] = (tmp["Rainfall_mm"] < 300).astype(int)
        tmp["wind_gt_10"] = (tmp["Wind_Speed_kmh"] > 10).astype(int)

        d = pd.get_dummies(
            tmp[
                [
                    "Crop_Growth_Stage",
                    "Mulching_Used",
                    "soil_lt_25",
                    "temp_gt_30",
                    "rain_lt_300",
                    "wind_gt_10",
                ]
            ],
            drop_first=False,
        )

        if return_dummy_cols:
            dummy_cols = d.columns.tolist()
        else:
            dummy_cols = self._logit_dummy_cols

        d = d.reindex(columns=dummy_cols, fill_value=0)

        z_low = (
            16.3173
            + (-11.0237 * d.get("soil_lt_25", 0))
            + (-5.8559 * d.get("temp_gt_30", 0))
            + (-10.8500 * d.get("rain_lt_300", 0))
            + (-5.8284 * d.get("wind_gt_10", 0))
            + (-5.4155 * d.get("Crop_Growth_Stage_Flowering", 0))
            + (5.5073 * d.get("Crop_Growth_Stage_Harvest", 0))
            + (5.2299 * d.get("Crop_Growth_Stage_Sowing", 0))
            + (-5.4617 * d.get("Crop_Growth_Stage_Vegetative", 0))
            + (-3.0014 * d.get("Mulching_Used_No", 0))
            + (2.8613 * d.get("Mulching_Used_Yes", 0))
        )

        z_med = (
            4.6524
            + (0.3290 * d.get("soil_lt_25", 0))
            + (-0.0204 * d.get("temp_gt_30", 0))
            + (0.1542 * d.get("rain_lt_300", 0))
            + (0.0841 * d.get("wind_gt_10", 0))
            + (0.3586 * d.get("Crop_Growth_Stage_Flowering", 0))
            + (-0.1348 * d.get("Crop_Growth_Stage_Harvest", 0))
            + (-0.3547 * d.get("Crop_Growth_Stage_Sowing", 0))
            + (0.3334 * d.get("Crop_Growth_Stage_Vegetative", 0))
            + (0.1883 * d.get("Mulching_Used_No", 0))
            + (0.0142 * d.get("Mulching_Used_Yes", 0))
        )

        z_high = (
            -20.9697
            + (10.6947 * d.get("soil_lt_25", 0))
            + (5.8763 * d.get("temp_gt_30", 0))
            + (10.6958 * d.get("rain_lt_300", 0))
            + (5.7444 * d.get("wind_gt_10", 0))
            + (5.0569 * d.get("Crop_Growth_Stage_Flowering", 0))
            + (-5.3725 * d.get("Crop_Growth_Stage_Harvest", 0))
            + (-4.8752 * d.get("Crop_Growth_Stage_Sowing", 0))
            + (5.1283 * d.get("Crop_Growth_Stage_Vegetative", 0))
            + (2.8131 * d.get("Mulching_Used_No", 0))
            + (-2.8755 * d.get("Mulching_Used_Yes", 0))
        )

        out = pd.DataFrame(
            {
                "logit_low": z_low,
                "logit_med": z_med,
                "logit_high": z_high,
            },
            index=df.index,
        )

        if return_dummy_cols:
            return out, dummy_cols
        return out

    def _fit_frequency_encoding(self, X):
        self._fe_cols = list(dict.fromkeys(self.cat_features_))
        self._freq_encodings = {}
        for c in self._fe_cols:
            if c in X.columns:
                self._freq_encodings[c] = X[c].value_counts(normalize=True).to_dict()

    def _apply_frequency_encoding(self, X):
        X = X.copy()
        for c, mapping in self._freq_encodings.items():
            if c in X.columns:
                X[f"{c}_fe"] = X[c].map(mapping).astype(float).fillna(0)
        return X

    def _encode_cat_codes(self, df):
        df = df.copy()
        for col, t in self._cat_types.items():
            if col in df.columns:
                s = df[col].astype("string").fillna("__NA__").astype(t)
                df[col] = s.cat.codes.astype("int32").astype("category")
        return df


X, y, test_X, cat_features, num_features = Preprocessing(
    X_base, y, test_base, orig
).fit_transform()

print("X shape:", X.shape)
print("test shape:", test_X.shape)
print("n_cat_features:", len(cat_features))
print("n_num_features:", len(num_features))

X shape: (630000, 165)
test shape: (270000, 165)
n_cat_features: 124
n_num_features: 41


In [8]:
# ============================================================
# Train CAT2_8
# ============================================================

skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

oof = np.zeros((len(X), 3), dtype=np.float32)
pred = np.zeros((len(test_X), 3), dtype=np.float32)

fold_scores = []
sample_weights_all = compute_sample_weight(class_weight="balanced", y=y)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("\n" + "=" * 60)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 60)

    X_tr = X.iloc[tr_idx].copy().reset_index(drop=True)
    X_va = X.iloc[va_idx].copy().reset_index(drop=True)
    X_te = test_X.copy().reset_index(drop=True)

    y_tr = y.iloc[tr_idx].reset_index(drop=True)
    y_va = y.iloc[va_idx].reset_index(drop=True)
    w_tr = sample_weights_all[tr_idx]

    te_cols = cat_features + num_features

    def rename_te(arr):
        return pd.DataFrame(arr, columns=[f"te_{i}" for i in range(arr.shape[1])])

    te = TargetEncoder(
        random_state=CFG.SEED,
        shuffle=True,
        cv=10,
        smooth="auto",
        target_type="multiclass",
    )

    X_tr_enc = rename_te(te.fit_transform(X_tr[te_cols], y_tr).astype("float32"))
    X_va_enc = rename_te(te.transform(X_va[te_cols]).astype("float32"))
    X_te_enc = rename_te(te.transform(X_te[te_cols]).astype("float32"))

    X_tr2 = pd.concat([X_tr.reset_index(drop=True), X_tr_enc], axis=1)
    X_va2 = pd.concat([X_va.reset_index(drop=True), X_va_enc], axis=1)
    X_te2 = pd.concat([X_te.reset_index(drop=True), X_te_enc], axis=1)

    # 共有コードの意図に寄せて、元カテゴリ列は落とす
    # CatBoostには数値化済み特徴 + TE特徴を食わせる
    X_tr2 = X_tr2.drop(columns=cat_features, errors="ignore")
    X_va2 = X_va2.drop(columns=cat_features, errors="ignore")
    X_te2 = X_te2.drop(columns=cat_features, errors="ignore")

    for df in (X_tr2, X_va2, X_te2):
        for c in df.columns:
            if str(df[c].dtype) == "category":
                df[c] = df[c].astype("int32")

    params = CFG.CAT_PARAMS.copy()

    model = CatBoostClassifier(**params)
    model.fit(
        X_tr2,
        y_tr,
        sample_weight=w_tr,
        eval_set=(X_va2, y_va),
    )

    va_pred = model.predict_proba(X_va2).astype(np.float32)
    te_pred = model.predict_proba(X_te2).astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_score = score_ba(y_va, va_pred)
    fold_scores.append(float(fold_score))
    print(f"fold BA: {fold_score:.7f}")

    del X_tr, X_va, X_te, X_tr2, X_va2, X_te2, X_tr_enc, X_va_enc, X_te_enc, model
    gc.collect()


raw_cv = score_ba(y, oof)
print("\nraw CV:", raw_cv)
print("fold_scores:", fold_scores)
print("mean fold:", float(np.mean(fold_scores)))


Fold 1/10
0:	learn: 1.0113195	test: 1.0093974	best: 1.0093974 (0)	total: 19.1s	remaining: 4d 10h 15m 48s
500:	learn: 0.0685806	test: 0.0643590	best: 0.0643590 (500)	total: 25.3s	remaining: 16m 23s
1000:	learn: 0.0618125	test: 0.0601803	best: 0.0601803 (1000)	total: 31.3s	remaining: 9m 54s
1500:	learn: 0.0579789	test: 0.0583451	best: 0.0583450 (1499)	total: 37.4s	remaining: 7m 41s
2000:	learn: 0.0550583	test: 0.0569315	best: 0.0569315 (2000)	total: 43.5s	remaining: 6m 31s
2500:	learn: 0.0528539	test: 0.0560842	best: 0.0560842 (2500)	total: 49.5s	remaining: 5m 46s
3000:	learn: 0.0509827	test: 0.0553707	best: 0.0553707 (3000)	total: 55.6s	remaining: 5m 14s
3500:	learn: 0.0493915	test: 0.0547563	best: 0.0547563 (3500)	total: 1m 1s	remaining: 4m 49s
4000:	learn: 0.0479754	test: 0.0542591	best: 0.0542590 (3997)	total: 1m 7s	remaining: 4m 29s
4500:	learn: 0.0467096	test: 0.0538461	best: 0.0538461 (4500)	total: 1m 13s	remaining: 4m 11s
5000:	learn: 0.0455590	test: 0.0534618	best: 0.0534600 (4

In [9]:
# ============================================================
# Threshold / class multiplier optimization
# ============================================================

def neg_balanced_accuracy(weights):
    adjusted = oof * np.asarray(weights).reshape(1, -1)
    return -balanced_accuracy_score(y, adjusted.argmax(axis=1))


result = differential_evolution(
    neg_balanced_accuracy,
    bounds=[(0, 3.0), (0, 3.0), (0, 3.0)],
    seed=CFG.SEED,
    maxiter=1000,
    tol=1e-8,
    polish=True,
)

best_weights = result.x
thresholded_cv = -result.fun

print("best_weights:", best_weights)
print("thresholded_cv:", thresholded_cv)

oof_thr = oof * best_weights.reshape(1, -1)
pred_thr = pred * best_weights.reshape(1, -1)

# blend用には行正規化版を推奨
oof_thr_norm = normalize_rows(oof_thr)
pred_thr_norm = normalize_rows(pred_thr)

print("thresholded_norm_cv:", score_ba(y, oof_thr_norm))

best_weights: [0.85315427 0.89558784 2.72593577]
thresholded_cv: 0.9798411062610158
thresholded_norm_cv: 0.9798411062610158


In [10]:
# ============================================================
# Save
# ============================================================

np.save(CFG.SAVE_DIR / "oof_mikhail_cat2_8_raw.npy", oof)
np.save(CFG.SAVE_DIR / "pred_mikhail_cat2_8_raw.npy", pred)

np.save(CFG.SAVE_DIR / "oof_mikhail_cat2_8_thresholded.npy", oof_thr.astype(np.float32))
np.save(CFG.SAVE_DIR / "pred_mikhail_cat2_8_thresholded.npy", pred_thr.astype(np.float32))

np.save(CFG.SAVE_DIR / "oof_mikhail_cat2_8_thresholded_norm.npy", oof_thr_norm)
np.save(CFG.SAVE_DIR / "pred_mikhail_cat2_8_thresholded_norm.npy", pred_thr_norm)

submission = sample.copy()
submission[CFG.TARGET] = pd.Series(np.argmax(pred_thr_norm, axis=1)).map(CFG.INV_TARGET_MAP)
submission.to_csv(CFG.SAVE_DIR / "submission_mikhail_cat2_8_thresholded_norm.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "raw_cv": float(raw_cv),
    "thresholded_cv": float(thresholded_cv),
    "thresholded_norm_cv": float(score_ba(y, oof_thr_norm)),
    "best_weights": [float(x) for x in best_weights],
    "fold_scores": fold_scores,
    "model": "CAT2_8",
    "source": "Mikhail shared code simplified/retrained",
    "notes": {
        "use_for_blend": "prefer thresholded_norm npy for LR stack",
        "raw_shared_reference": "CAT2_8 raw score around 0.9781817, thresholded CV around 0.9800430 in shared notebook",
    },
}

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

print("saved:", CFG.SAVE_DIR)

saved: /kaggle/working/exp_20260427_058_mikhail_cat2_8_repro


In [11]:
# ============================================================
# Correlation vs existing OOF
# ============================================================

corr_rows = []
for name, path in CFG.REF_OOF.items():
    if os.path.exists(path):
        ref = np.load(path)
        corr_rows.append({
            "ref": name,
            "raw_corr_mean": mean_raw_corr(oof_thr_norm, ref),
            "rank_corr_mean": mean_rank_corr(oof_thr_norm, ref),
        })

corr_df = pd.DataFrame(corr_rows).sort_values("rank_corr_mean", ascending=False)
corr_df.to_csv(CFG.SAVE_DIR / "corr_vs_existing_oof.csv", index=False)
display(corr_df)

print("Done.")

,ref,raw_corr_mean,rank_corr_mean
4,057,0.993444,0.961972
1,046b,0.993082,0.959005
2,025,0.993405,0.951290
3,056,0.990957,0.948458
0,018,0.988423,0.929263


Done.
